In [1]:
import torch
import torch.nn as nn
import numpy as np
import random
import json

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Project paths
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"

# Load saved vocabulary
with open(DATA_DIR / "vocab.json", encoding="utf-8") as f:
    word_to_id = json.load(f)

# Load encoded data
token_data = torch.load(
    DATA_DIR / "token_ids.pt",
    weights_only=True
)

train_ids = token_data["train"]
val_ids = token_data["validation"]
test_ids = token_data["test"]

VOCAB_SIZE = len(word_to_id)
PAD_ID = word_to_id["<pad>"]
UNK_ID = word_to_id["<unk>"]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Vocabulary size:", VOCAB_SIZE)
print("Training tokens:", len(train_ids))

Device: cpu
Vocabulary size: 10000
Training tokens: 2051910


In [2]:
class RNNLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        num_layers=1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):

        # x: (batch_size, sequence_length)
        embedded = self.embedding(x)

        # embedded: (batch_size, sequence_length, 100)
        rnn_output, hidden = self.rnn(embedded)

        # rnn_output: (batch_size, sequence_length, 128)
        logits = self.fc(rnn_output)

        # logits: (batch_size, sequence_length, vocab_size)
        return logits

In [3]:
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
NUM_LAYERS = 1

model = RNNLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
).to(DEVICE)

print(model)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

RNNLanguageModel(
  (embedding): Embedding(10000, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10000, bias=True)
)

Total parameters: 2,319,440
Trainable parameters: 2,319,440


In [4]:
dummy_input = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(32, 32)
).to(DEVICE)

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", dummy_output.shape)

assert dummy_output.shape == (32, 32, VOCAB_SIZE)

print("\nRNN forward pass successful!")

Input shape: torch.Size([32, 32])
Output shape: torch.Size([32, 32, 10000])

RNN forward pass successful!


In [5]:
from torch.utils.data import Dataset, DataLoader

CONTEXT_LENGTH = 32
BATCH_SIZE = 32

class LanguageModelDataset(Dataset):

    def __init__(self, token_ids, context_length=32):
        self.tokens = torch.tensor(token_ids, dtype=torch.long)
        self.context_length = context_length

    def __len__(self):
        return (len(self.tokens) - 1) // self.context_length

    def __getitem__(self, idx):
        start = idx * self.context_length
        end = start + self.context_length

        x = self.tokens[start:end]
        y = self.tokens[start + 1:end + 1]

        return x, y


train_dataset = LanguageModelDataset(train_ids, CONTEXT_LENGTH)
val_dataset = LanguageModelDataset(val_ids, CONTEXT_LENGTH)
test_dataset = LanguageModelDataset(test_ids, CONTEXT_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 2004
Validation batches: 209
Test batches: 236


In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

Loss function: CrossEntropyLoss()
Optimizer: Adam


In [7]:
model.train()

x_batch, y_batch = next(iter(train_loader))

x_batch = x_batch.to(DEVICE)
y_batch = y_batch.to(DEVICE)

optimizer.zero_grad()

# Forward pass
logits = model(x_batch)

# Reshape for cross-entropy
loss = criterion(
    logits.reshape(-1, VOCAB_SIZE),
    y_batch.reshape(-1)
)

# Backpropagation
loss.backward()

# Prevent exploding gradients
torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

# Update parameters
optimizer.step()

print(f"Initial training loss: {loss.item():.4f}")
print(f"Initial perplexity: {torch.exp(loss).item():.2f}")
print("One training step completed successfully!")

Initial training loss: 9.2624
Initial perplexity: 10534.71
One training step completed successfully!


In [8]:
import time
import torch

# Reset the model and optimizer for reproducibility
torch.manual_seed(SEED)

model = RNNLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

NUM_BENCHMARK_BATCHES = 20

model.train()

start_time = time.perf_counter()

for batch_idx, (x, y) in enumerate(train_loader):

    if batch_idx >= NUM_BENCHMARK_BATCHES:
        break

    x = x.to(DEVICE)
    y = y.to(DEVICE)

    optimizer.zero_grad()

    logits = model(x)

    loss = criterion(
        logits.reshape(-1, VOCAB_SIZE),
        y.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

elapsed = time.perf_counter() - start_time

seconds_per_batch = elapsed / NUM_BENCHMARK_BATCHES
estimated_epoch_time = seconds_per_batch * len(train_loader)

print(f"Benchmark batches: {NUM_BENCHMARK_BATCHES}")
print(f"Total benchmark time: {elapsed:.2f} seconds")
print(f"Average time per batch: {seconds_per_batch:.3f} seconds")
print(f"Estimated time per epoch: {estimated_epoch_time / 60:.2f} minutes")

Benchmark batches: 20
Total benchmark time: 1.87 seconds
Average time per batch: 0.094 seconds
Estimated time per epoch: 3.13 minutes


In [9]:
EPOCHS = 5
LEARNING_RATE = 0.001

# Reset random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Reinitialize model
model = RNNLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Store training history
history = {
    "train_loss": [],
    "val_loss": [],
    "epoch_time": []
}

print("Model reinitialized successfully!")

Model reinitialized successfully!


In [10]:
from tqdm.auto import tqdm
import time

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for x, y in tqdm(loader, desc="Training", leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        num_tokens = (y != PAD_ID).sum().item()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

    return total_loss / total_tokens


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for x, y in tqdm(loader, desc="Validation", leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        )

        num_tokens = (y != PAD_ID).sum().item()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

    return total_loss / total_tokens

In [11]:
from pathlib import Path
import math

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

best_val_loss = float("inf")
total_training_time = 0.0

for epoch in range(EPOCHS):

    start_time = time.perf_counter()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss = evaluate(
        model, val_loader, criterion
    )

    epoch_time = time.perf_counter() - start_time
    total_training_time += epoch_time

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["epoch_time"].append(epoch_time)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val PPL: {math.exp(val_loss):.2f} | "
        f"Time: {epoch_time:.1f}s"
    )

    # Save best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "val_loss": val_loss,
            "config": {
                "vocab_size": VOCAB_SIZE,
                "embedding_dim": EMBEDDING_DIM,
                "hidden_dim": HIDDEN_DIM,
                "num_layers": NUM_LAYERS
            }
        }, MODELS_DIR / "rnn_best.pt")

print("\nTraining completed!")
print(f"Total training time: {total_training_time / 60:.2f} minutes")
print(f"Best validation perplexity: {math.exp(best_val_loss):.2f}")

Training:   0%|          | 0/2004 [00:00<?, ?it/s]

Validation:   0%|          | 0/209 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 5.7004 | Val Loss: 5.2490 | Val PPL: 190.38 | Time: 290.9s


Training:   0%|          | 0/2004 [00:00<?, ?it/s]

Validation:   0%|          | 0/209 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/209 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 4.8371 | Val Loss: 4.9731 | Val PPL: 144.47 | Time: 280.0s


Training:   0%|          | 0/2004 [00:00<?, ?it/s]

Validation:   0%|          | 0/209 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 4.7449 | Val Loss: 4.9366 | Val PPL: 139.30 | Time: 276.7s

Training completed!
Total training time: 23.56 minutes
Best validation perplexity: 139.30


In [12]:
checkpoint = torch.load(
    MODELS_DIR / "rnn_best.pt",
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best model loaded successfully!")
print("Best epoch:", checkpoint["epoch"])
print(f"Validation loss: {checkpoint['val_loss']:.4f}")

Best model loaded successfully!
Best epoch: 5
Validation loss: 4.9366


In [13]:
import math
import time

start_time = time.perf_counter()

test_loss = evaluate(
    model,
    test_loader,
    criterion
)

test_time = time.perf_counter() - start_time
test_ppl = math.exp(test_loss)

print(f"Test cross-entropy: {test_loss:.4f}")
print(f"Test perplexity: {test_ppl:.2f}")
print(f"Test evaluation time: {test_time:.2f} seconds")

Validation:   0%|          | 0/236 [00:00<?, ?it/s]

Test cross-entropy: 4.8550
Test perplexity: 128.39
Test evaluation time: 4.83 seconds


In [14]:
import torch

id_to_word = {
    idx: word for word, idx in word_to_id.items()
}

@torch.no_grad()
def generate_rnn(
    model,
    prompt,
    max_new_tokens=30,
    temperature=1.0,
    top_k=20,
    seed=42
):
    model.eval()

    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)

    tokens = prompt.lower().split()

    generated_ids = [
        word_to_id.get(token, UNK_ID)
        for token in tokens
    ]

    for _ in range(max_new_tokens):

        # Use at most the last 32 tokens
        input_ids = generated_ids[-CONTEXT_LENGTH:]

        x = torch.tensor(
            [input_ids],
            dtype=torch.long,
            device=DEVICE
        )

        # Forward pass
        logits = model(x)

        # Next-token scores
        next_logits = logits[0, -1, :].clone()

        # Exclude padding token
        next_logits[PAD_ID] = float("-inf")

        # Temperature scaling
        next_logits = next_logits / temperature

        # Top-k filtering
        if top_k is not None:
            values, indices = torch.topk(
                next_logits, k=top_k
            )

            filtered_logits = torch.full_like(
                next_logits,
                float("-inf")
            )

            filtered_logits[indices] = values
            next_logits = filtered_logits

        # Convert scores to probabilities
        probs = torch.softmax(next_logits, dim=-1)

        # Sample next token
        next_id = torch.multinomial(
            probs,
            num_samples=1,
            generator=generator
        ).item()

        generated_ids.append(next_id)

    return " ".join(
        id_to_word[idx] for idx in generated_ids
    )

In [15]:
prompts = [
    "the game",
    "in the",
    "the history of"
]

for prompt in prompts:

    generated_text = generate_rnn(
        model=model,
        prompt=prompt,
        max_new_tokens=30,
        temperature=1.0,
        top_k=20,
        seed=42
    )

    print("=" * 60)
    print("PROMPT:", prompt)
    print("GENERATED:")
    print(generated_text)
    print()

PROMPT: the game
GENERATED:
the game , where he was unable to gain an important role , and a <unk> <unk> on his first appearance of the best @-@ selling album , though a song won

PROMPT: in the
GENERATED:
in the <unk> region . the <unk> also had been a only <unk> <unk> . the <unk> was a 4 – 1 – 0 away from <unk> to the east . this

PROMPT: the history of
GENERATED:
the history of <unk> . during this , the storm may <unk> in <unk> and <unk> <unk> . the storm attained heavy rainfall from the north pacific fleet , and a hurricane force



In [16]:
import json

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

rnn_results = {
    "model": "Vanilla RNN",
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "parameters": total_params,
    "epochs": EPOCHS,
    "best_epoch": checkpoint["epoch"],
    "best_val_loss": checkpoint["val_loss"],
    "best_val_perplexity": math.exp(checkpoint["val_loss"]),
    "test_loss": test_loss,
    "test_perplexity": test_ppl,
    "test_evaluation_time_seconds": test_time,
    "training_time_seconds": total_training_time,
    "history": history
}

with open(RESULTS_DIR / "rnn_results.json", "w") as f:
    json.dump(rnn_results, f, indent=2)

print("RNN results saved successfully!")

RNN results saved successfully!
